# Explore an Example NISAR GUNW Time Series with Tropospheric Correction

<br>

This notebook introduces the analysis of multi-temporal SAR interferogram data stacks using the NISAR L2 GUNW (Geocoded UNWrapped phase interferogram) data product. GUNW data are a geocoded product that provides unwrapped phase measurements from interferometric SAR acquisitions, along with supporting data layers including coherence, connected components, and wrapped phase.

**This notebook demonstrates:**
1. **Searching for GUNW data by track, date range, and area of interest**
2. **Downloading GUNW products on-the-fly using asf_search**
3. Understanding NISAR GUNW product structure and available data layers
4. Accessing and loading GUNW data from cloud storage or local files
5. Extracting unwrapped phase and coherence data
6. Loading and applying OPERA TROPO atmospheric correction products
7. Creating time series stacks of corrected phase data
8. Visualizing unwrapped phase and coherence
9. Basic time-series analysis and deformation detection

<img src="https://asfopensarlab.github.io/opensarlab-notebook-assets/notebook_images/insar_geometry.jpeg" width="400" align="right" /> 

Interferometric SAR (InSAR) measures phase differences between two radar acquisitions to detect surface deformation. GUNW products represent the next generation of InSAR data products optimized for studying ground deformation at local to regional scales.

:::{attention} Notes about GUNW and TROPO data

NISAR GUNW products are newly released. This notebook uses example interferogram pairs selected for demonstration purposes.

OPERA TROPO (TROPOspheric) correction products provide atmospheric delay corrections derived from weather models. When available for your study area and acquisition date, TROPO corrections can significantly improve the accuracy of deformation measurements by removing atmospheric artifacts.

:::

<hr>

## Overview
1. [Prerequisites](gunw-ts-prereqs)
2. [Search and Download GUNW Data by Track](gunw-ts-search-download)
3. [NISAR GUNW Product Structure](gunw-ts-structure)
4. [Load and Explore GUNW Data](gunw-ts-load)
5. [Understanding Atmospheric Corrections with TROPO](gunw-ts-tropo)
6. [Load and Apply TROPO Corrections](gunw-ts-apply-tropo)
7. [Build and Visualize Time Series Stack](gunw-ts-timeseries)
8. [Basic Time-Series Analysis](gunw-ts-analysis)
9. [Summary and Next Steps](gunw-ts-summary)
10. [References](gunw-ts-references)

<hr>

(gunw-ts-prereqs)=
## 1. Prerequisites

| Prerequisite | Importance | Notes |
| --- | --- | --- |
| [The isce3 software environment for this cookbook](software_environment_docker.ipynb) | Necessary | |
| [How to search and access NISAR data in your area of interest](NISAR_data_access.ipynb) | Necessary | If you wish to process data in your AOI instead of the example |
| [Familiarity with xarray](https://docs.xarray.dev/en/stable/) | Helpful | For working with multi-dimensional data |
| [Familiarity with matplotlib and cartopy](https://matplotlib.org/stable/index.html) | Helpful | For visualization |
| [Understanding of interferometric SAR (InSAR)](https://nisar-docs.asf.alaska.edu/) | Helpful | Background on InSAR concepts |

### Required Python Packages
- `asf_search`: For searching and downloading NISAR data from ASF
- `xarray`, `rioxarray`: For working with multi-dimensional data
- `h5py`: For reading HDF5 files
- `matplotlib`, `cartopy`: For visualization

- **Rough Notebook Time Estimate**: 30-45 minutes
 
<hr>

(gunw-ts-search-download)=
## 2. Search and Download GUNW Data by Track

This section demonstrates how to search for GUNW data by track number, date range, and other parameters, then download the files automatically.

**Reference**: [ASF Search Documentation](https://nisar-docs.asf.alaska.edu/asf-search/)

### 2a. Import required libraries

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import asf_search as asf
from datetime import datetime, timedelta
import numpy as np
import xarray as xr
import rioxarray
import h5py
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import SymLogNorm
import pandas as pd
from pathlib import Path
from getpass import getpass

print("Libraries imported successfully!")

### 2b. Authenticate with Earthdata Login

In [ ]:
# Authenticate with Earthdata Login
# You need an Earthdata account: https://urs.earthdata.nasa.gov/
# Add "ARIA Product Search" to your authorized applications

print("Setting up Earthdata authentication...")
print("If you don't have an account, go to: https://urs.earthdata.nasa.gov/")
print()

try:
    session = asf.ASFSession()
    username = input("Enter Earthdata Login username: ")
    password = getpass("Enter Earthdata Login password: ")
    session.auth_with_creds(username, password)
    print("✓ Authentication successful!")
except Exception as e:
    print(f"Authentication failed: {e}")
    print("You can still search without authentication, but downloads may be limited.")

### 2c. Define search parameters for GUNW data

In [ ]:
# =============================================================================
# CONFIGURE THESE PARAMETERS FOR YOUR STUDY AREA
# =============================================================================

# Track number (0-175 for NISAR L-band, varies by mission phase)
# Find track numbers for your area: https://search.asf.alaska.edu/
track_number = 42  # Example track

# Frame number (within track) - optional, use None to get all frames
frame_number = 10  # Example frame (use None to get all)

# Flight direction: 'ASCENDING' or 'DESCENDING'
flight_direction = 'ASCENDING'  # or 'DESCENDING'

# Date range for search
start_date = datetime(2024, 1, 1)
end_date = datetime(2024, 12, 31)

# Area of interest (WKT polygon format)
# This is optional - if not provided, searches entire track
# Format: "POLYGON((lon1 lat1, lon2 lat2, lon3 lat3, lon4 lat4, lon1 lat1))"
# Leave as None to search entire track, or specify your AOI:
aoi = None  # Set to None for entire track, or define polygon

# Output directory for downloads
output_dir = Path.home() / "NISAR_GUNW_data"
output_dir.mkdir(exist_ok=True)

# Maximum number of results to return
max_results = 100

print(f"Search Parameters:")
print(f"  Track: {track_number}")
print(f"  Frame: {frame_number if frame_number else 'All'}")
print(f"  Flight Direction: {flight_direction}")
print(f"  Date Range: {start_date.date()} to {end_date.date()}")
print(f"  AOI: {aoi if aoi else 'Entire track'}")
print(f"  Output Directory: {output_dir}")
print()

### 2d. Search for GUNW products

In [ ]:
print(f"Searching for GUNW products on track {track_number}...\n")

# Build search options
search_opts = {
    "processingLevel": ["GUNW"],           # GUNW = Geocoded UNWrapped interferograms
    "dataset": ["NISAR"],                  # NISAR mission
    "productionConfiguration": ["PR"],    # PR = Production Ready
    "maxResults": max_results,
    "relativeOrbit": track_number,         # Track/relative orbit number
    "flightDirection": flight_direction,
    "start": start_date,
    "end": end_date,
    "session": session if 'session' in locals() else None,
}

# Add frame if specified
if frame_number is not None:
    search_opts["frame"] = frame_number

# Add AOI if specified
if aoi is not None:
    search_opts["intersectsWith"] = aoi

# Remove None session if not authenticated
if search_opts["session"] is None:
    del search_opts["session"]

try:
    # Perform search
    opts = asf.ASFSearchOptions(**search_opts)
    response = asf.search(opts=opts)
    
    # Extract results
    gunw_products = response.results()
    
    print(f"✓ Found {len(gunw_products)} GUNW products\n")
    
    if len(gunw_products) > 0:
        # Create DataFrame for better visualization
        product_data = []
        for product in gunw_products:
            product_data.append({
                'Filename': product.properties.get('filename', 'N/A'),
                'Start Time': product.properties.get('startTime', 'N/A'),
                'Stop Time': product.properties.get('stopTime', 'N/A'),
                'Track': product.properties.get('pathNumber', 'N/A'),
                'Frame': product.properties.get('frameNumber', 'N/A'),
            })
        
        df_products = pd.DataFrame(product_data)
        print(df_products.to_string(index=False))
        print()
    else:
        print("No GUNW products found for your search parameters.")
        print("Try adjusting track number, dates, or other parameters.")

except Exception as e:
    print(f"Search error: {e}")
    print("\nTroubleshooting:")
    print("1. Check your internet connection")
    print("2. Verify Earthdata credentials")
    print("3. Try different track/frame/date parameters")
    print("4. Visit https://search.asf.alaska.edu/ to find available data")

### 2e. Extract download URLs and filter by coherence/quality (optional)

In [ ]:
# Extract HDF5 download URLs
# Filter out QA_STATS files (metadata, not actual data)

if len(gunw_products) > 0:
    pattern = r'^(?!.*QA_STATS).*'  # Regex to exclude QA_STATS files
    
    h5_urls = response.find_urls(extension='.h5', pattern=pattern, directAccess=False)
    
    print(f"Found {len(h5_urls)} HDF5 data files (excluding QA_STATS)\n")
    
    # Display first few URLs
    print("Sample download URLs:")
    for i, url in enumerate(h5_urls[:3]):
        # Shorten URL for display
        display_url = "..." + url[-80:] if len(url) > 80 else url
        print(f"  {i+1}. {display_url}")
    
    if len(h5_urls) > 3:
        print(f"  ... and {len(h5_urls) - 3} more")
else:
    h5_urls = []
    print("No files to download (no products found)")

### 2f. Download GUNW files

In [ ]:
# Download the GUNW products
# IMPORTANT: GUNW files are typically 1-5 GB each. Make sure you have adequate disk space.

if len(h5_urls) > 0:
    print(f"Downloading {len(h5_urls)} GUNW files to {output_dir}...\n")
    print("WARNING: This may take a while depending on file sizes and internet speed.")
    print()
    
    try:
        # Download with multiple processes for faster download
        # Adjust 'processes' parameter based on your connection speed
        downloaded_files = asf.download_urls(
            h5_urls, 
            output_dir, 
            session=session if 'session' in locals() else None,
            processes=2  # Use 2 parallel downloads to avoid overwhelming the server
        )
        
        print(f"\n✓ Download complete!")
        print(f"Downloaded files: {len(downloaded_files)}")
        
        # List downloaded files
        downloaded_gunw_files = list(output_dir.glob("NISAR_L2_*_GUNW*.h5"))
        print(f"\nGUNW files in {output_dir}:")
        for i, f in enumerate(sorted(downloaded_gunw_files), 1):
            size_mb = f.stat().st_size / (1024 * 1024)
            print(f"  {i}. {f.name} ({size_mb:.1f} MB)")
    
    except Exception as e:
        print(f"Download error: {e}")
        print("\nPossible solutions:")
        print("1. Check your Earthdata credentials")
        print("2. Ensure 'ARIA Product Search' is authorized in your Earthdata account")
        print("3. Try downloading fewer files at a time")
else:
    print("No files to download")

<hr>

(gunw-ts-structure)=
## 3. NISAR GUNW Product Structure

NISAR GUNW products contain the following main data layers:

### Key Data Layers:
- **unwrappedPhase**: Phase (in radians) that has been unwrapped to remove 2π discontinuities
- **wrappedInterferogram**: Raw phase before unwrapping (wrapped to [-π, π])
- **coherence** (or coherenceMagnitude): Measure of phase quality (0=poor, 1=excellent)
- **connectedComponents**: Labels for phase unwrapping regions, useful for identifying unwrapping errors
- **ionospherePhaseScreen**: Estimated ionospheric delay (when available)
- **slantRangeOffset** and **alongTrackOffset**: Pixel offsets from coregistration

### Product Organization:
GUNW products are organized in HDF5 format with the following structure:
```
/science/LSAR/GUNW/
  ├── metadata/
  │   └── radarGrid/
  └── grids/
      ├── frequencyA/
      │   ├── unwrappedInterferogram/{pol}/
      │   ├── wrappedInterferogram/{pol}/
      │   └── pixelOffsets/{pol}/
      └── frequencyB/ (if available)
```

where {pol} represents polarization (e.g., 'HH', 'VV', 'HV', 'VH')

<hr>

(gunw-ts-load)=
## 4. Load and Explore GUNW Data

### 4a. Function to inspect GUNW file structure

In [ ]:
def inspect_gunw_hdf5(filepath):
    """
    Inspect the structure and available datasets in a GUNW HDF5 file.
    
    Parameters:
    -----------
    filepath : str
        Path to the GUNW HDF5 file
    """
    print(f"Inspecting: {filepath}\n")
    
    def print_structure(name, obj, depth=0):
        indent = "  " * depth
        if isinstance(obj, h5py.Dataset):
            print(f"{indent}Dataset: {name}")
            print(f"{indent}  Shape: {obj.shape}, Dtype: {obj.dtype}")
        elif isinstance(obj, h5py.Group):
            print(f"{indent}Group: {name}/")
    
    try:
        with h5py.File(filepath, 'r') as f:
            f.visititems(print_structure)
    except FileNotFoundError:
        print(f"File not found: {filepath}")
        print("Download GUNW files first using the search/download cells above.")

# Example usage (uncomment and modify path as needed):
# gunw_files = list(output_dir.glob("NISAR_L2_*_GUNW*.h5"))
# if gunw_files:
#     inspect_gunw_hdf5(str(gunw_files[0]))

### 4b. Function to load GUNW data layers

In [ ]:
def load_gunw_data(filepath, polarization='VV', data_type='unwrappedPhase'):
    """
    Load specific data layers from a GUNW HDF5 file.
    
    Parameters:
    -----------
    filepath : str
        Path to the GUNW HDF5 file
    polarization : str
        Polarization to load ('VV', 'HH', 'HV', 'VH')
    data_type : str
        Type of data to load ('unwrappedPhase', 'wrappedInterferogram', 'coherence', etc.)
    
    Returns:
    --------
    tuple : (data, geotransform, projection) - data array, geographic info
    """
    
    # Determine the HDF5 path based on data type
    if data_type in ['unwrappedPhase', 'coherence', 'connectedComponents', 'ionospherePhaseScreen']:
        group_path = f'/science/LSAR/GUNW/grids/frequencyA/unwrappedInterferogram/{polarization}/{data_type}'
    elif data_type in ['wrappedInterferogram']:
        group_path = f'/science/LSAR/GUNW/grids/frequencyA/wrappedInterferogram/{polarization}/{data_type}'
    else:
        raise ValueError(f"Unknown data_type: {data_type}")
    
    try:
        with h5py.File(filepath, 'r') as f:
            # Load data
            if group_path in f:
                data = f[group_path][:]
                print(f"Loaded {data_type} for polarization {polarization}")
                print(f"  Shape: {data.shape}")
                print(f"  Data type: {data.dtype}")
                print(f"  Min: {np.nanmin(data):.4f}, Max: {np.nanmax(data):.4f}")
                return data
            else:
                print(f"Dataset not found at {group_path}")
                return None
    except FileNotFoundError:
        print(f"File not found: {filepath}")
        return None

print("Load functions defined successfully!")

<hr>

(gunw-ts-tropo)=
## 5. Understanding Atmospheric Corrections with TROPO

### Background on Atmospheric Effects in InSAR

Atmospheric delays are a major error source in interferometric SAR measurements. Two types of delays affect radar signals:

1. **Tropospheric delay**: Caused by variations in pressure, temperature, and humidity in the lower atmosphere (0-10 km)
   - Typically the largest atmospheric error (~50-100 mm in LOS)
   - Highly correlated with topography and weather patterns

2. **Ionospheric delay**: Caused by free electrons in the ionosphere (>50 km altitude)
   - Typically smaller than tropospheric delay
   - Frequency-dependent (worse at lower frequencies like L-band)

### OPERA TROPO Product

The OPERA project provides **TROPO** (TROPOspheric) correction products that:
- Estimate tropospheric delays using weather models (HRRR, ERA5, etc.)
- Provide zenith delay estimates that must be projected to slant-range
- Reduce tropospheric noise by ~40-50% when applied correctly

**Reference**: https://www.jpl.nasa.gov/go/opera/products/tropo-product/

### Converting Tropospheric Correction to Phase

Tropospheric delay (in meters) is converted to phase using:
```
Phase_correction (radians) = -4π / λ × delay (m)
```

where λ is the radar wavelength (NISAR L-band: ~23.8 cm)

<hr>

(gunw-ts-apply-tropo)=
## 6. Load and Apply TROPO Corrections

### 6a. Define NISAR parameters and utility functions

In [ ]:
# NISAR L-band parameters
NISAR_L_WAVELENGTH = 0.2377  # meters (24 GHz center frequency)
SPEED_OF_LIGHT = 299792458   # m/s

def delay_to_phase(delay_m, wavelength=NISAR_L_WAVELENGTH):
    """
    Convert atmospheric delay (in meters) to phase (in radians).
    
    The relationship is:
    Phase_correction = -4π / λ × delay
    
    The negative sign indicates that a positive (wet) tropospheric delay
    causes a phase advance (negative phase change in the InSAR sense).
    
    Parameters:
    -----------
    delay_m : float or np.ndarray
        Atmospheric delay in meters
    wavelength : float
        Radar wavelength in meters (default: NISAR L-band)
    
    Returns:
    --------
    float or np.ndarray : Phase in radians
    """
    return -4 * np.pi / wavelength * delay_m

def phase_to_delay(phase_rad, wavelength=NISAR_L_WAVELENGTH):
    """
    Convert phase (in radians) to atmospheric delay (in meters).
    Inverse of delay_to_phase().
    """
    return -wavelength / (4 * np.pi) * phase_rad

def apply_tropospheric_correction(unwrapped_phase, tropo_delay_m):
    """
    Apply tropospheric correction to unwrapped interferometric phase.
    
    Parameters:
    -----------
    unwrapped_phase : np.ndarray
        Unwrapped phase in radians
    tropo_delay_m : np.ndarray
        Tropospheric delay in meters (differential: secondary - reference)
    
    Returns:
    --------
    np.ndarray : Corrected phase in radians
    """
    tropo_phase_correction = delay_to_phase(tropo_delay_m)
    corrected_phase = unwrapped_phase - tropo_phase_correction
    return corrected_phase

print(f"NISAR L-band wavelength: {NISAR_L_WAVELENGTH} m")
print(f"Conversion factor (4π/λ): {4*np.pi/NISAR_L_WAVELENGTH:.2f} rad/m")

### 6b. Function to load TROPO data

In [ ]:
def load_tropo_data(tropo_filepath, polarization='VV'):
    """
    Load tropospheric delay correction from OPERA TROPO product.
    
    OPERA TROPO products typically contain:
    - ZenithDelay for reference and secondary dates
    - Possibly also separated hydrostatic and wet components
    
    This function attempts to load differential tropospheric delay
    (secondary - reference).
    
    Parameters:
    -----------
    tropo_filepath : str
        Path to TROPO NetCDF or HDF5 file
    polarization : str
        Polarization identifier (for organizational purposes)
    
    Returns:
    --------
    np.ndarray or None : Tropospheric delay in meters
    """
    
    print(f"Loading TROPO data from: {tropo_filepath}")
    
    try:
        # Try loading as NetCDF (common TROPO format)
        if tropo_filepath.endswith('.nc'):
            import netCDF4
            with netCDF4.Dataset(tropo_filepath, 'r') as ds:
                # List available variables
                print(f"Available variables: {list(ds.variables.keys())}")
                
                # Try to load differential zenith delay
                if 'troposphericDelay' in ds.variables:
                    tropo_delay = ds['troposphericDelay'][:]
                    print(f"Loaded troposphericDelay: shape={tropo_delay.shape}")
                    return tropo_delay
                elif 'zenithDelay' in ds.variables:
                    zenith_delay = ds['zenithDelay'][:]
                    print(f"Loaded zenithDelay: shape={zenith_delay.shape}")
                    # Note: zenithDelay needs to be converted to slant delay using incidence angle
                    return zenith_delay
                else:
                    print("Warning: No recognized tropospheric delay variable found")
                    return None
        
        # Try loading as HDF5
        elif tropo_filepath.endswith('.h5') or tropo_filepath.endswith('.hdf5'):
            with h5py.File(tropo_filepath, 'r') as f:
                print(f"Available datasets: {list(f.keys())}")
                
                if 'troposphericDelay' in f:
                    tropo_delay = f['troposphericDelay'][:]
                    print(f"Loaded troposphericDelay: shape={tropo_delay.shape}")
                    return tropo_delay
                else:
                    print("Warning: No troposphericDelay dataset found")
                    return None
    
    except Exception as e:
        print(f"Error loading TROPO data: {e}")
        return None

print("TROPO loading function defined!")

<hr>

(gunw-ts-timeseries)=
## 7. Build and Visualize Time Series Stack

### 7a. Example workflow for creating a time-series stack

In [ ]:
class GUNWTimeSeries:
    """
    Class for managing and analyzing NISAR GUNW time-series data.
    """
    
    def __init__(self, name='GUNW_TimeSeries'):
        self.name = name
        self.gunw_files = []
        self.dates = []
        self.phase_stack = None
        self.coherence_stack = None
        self.tropo_corrections = []
        
    def add_interferogram(self, filepath, ref_date, sec_date, tropo_filepath=None):
        """
        Add a GUNW interferogram to the time series.
        
        Parameters:
        -----------
        filepath : str
            Path to GUNW HDF5 file
        ref_date : datetime
            Reference acquisition date
        sec_date : datetime
            Secondary acquisition date
        tropo_filepath : str, optional
            Path to corresponding TROPO correction file
        """
        self.gunw_files.append({
            'path': filepath,
            'ref_date': ref_date,
            'sec_date': sec_date,
            'tropo_path': tropo_filepath
        })
        self.dates.append((ref_date, sec_date))
        print(f"Added interferogram: {ref_date.date()} - {sec_date.date()}")
    
    def load_data_stack(self, polarization='VV'):
        """
        Load and stack phase and coherence data from all interferograms.
        """
        n_ifg = len(self.gunw_files)
        
        print(f"Loading {n_ifg} interferograms...\n")
        
        phase_list = []
        coherence_list = []
        
        for i, ifg_info in enumerate(self.gunw_files):
            print(f"Loading interferogram {i+1}/{n_ifg}: {ifg_info['ref_date'].date()} - {ifg_info['sec_date'].date()}")
            
            # Load unwrapped phase
            phase = load_gunw_data(ifg_info['path'], polarization=polarization, data_type='unwrappedPhase')
            
            if phase is not None:
                # Load coherence
                coherence = load_gunw_data(ifg_info['path'], polarization=polarization, data_type='coherence')
                
                # Apply TROPO correction if available
                if ifg_info['tropo_path'] is not None:
                    tropo_delay = load_tropo_data(ifg_info['tropo_path'], polarization=polarization)
                    if tropo_delay is not None:
                        phase = apply_tropospheric_correction(phase, tropo_delay)
                        print(f"  Applied TROPO correction (max delay: {np.nanmax(np.abs(tropo_delay)):.3f} m)")
                
                phase_list.append(phase)
                coherence_list.append(coherence)
                print(f"  ✓ Loaded successfully\n")
            else:
                print(f"  ✗ Failed to load\n")
        
        # Create stacks if data was loaded
        if phase_list:
            # Find common shape
            shapes = [p.shape for p in phase_list]
            min_shape = tuple(min(s[i] for s in shapes) for i in range(len(shapes[0])))
            
            # Crop all arrays to common shape
            phase_cropped = [p[:min_shape[0], :min_shape[1]] for p in phase_list]
            coherence_cropped = [c[:min_shape[0], :min_shape[1]] for c in coherence_list]
            
            # Stack into 3D arrays (n_ifg, height, width)
            self.phase_stack = np.stack(phase_cropped, axis=0)
            self.coherence_stack = np.stack(coherence_cropped, axis=0)
            
            print(f"\nStacks created:")
            print(f"  Phase stack shape: {self.phase_stack.shape}")
            print(f"  Coherence stack shape: {self.coherence_stack.shape}")
            print(f"  Phase range: [{np.nanmin(self.phase_stack):.2f}, {np.nanmax(self.phase_stack):.2f}] rad")
            print(f"  Coherence range: [{np.nanmin(self.coherence_stack):.3f}, {np.nanmax(self.coherence_stack):.3f}]")
        else:
            print("No data was successfully loaded")
    
    def plot_single_interferogram(self, ifg_index=0, cmap='RdBu_r', figsize=(14, 5)):
        """
        Plot unwrapped phase and coherence for a single interferogram.
        """
        if self.phase_stack is None:
            print("No data loaded. Run load_data_stack() first.")
            return
        
        if ifg_index >= len(self.dates):
            print(f"Invalid index. Available interferograms: 0-{len(self.dates)-1}")
            return
        
        fig, axes = plt.subplots(1, 3, figsize=figsize)
        
        ref_date, sec_date = self.dates[ifg_index]
        title_str = f'{ref_date.date()} - {sec_date.date()}'
        
        # Unwrapped phase
        im1 = axes[0].imshow(self.phase_stack[ifg_index, :, :], cmap=cmap)
        axes[0].set_title(f'Unwrapped Phase\n{title_str}')
        axes[0].set_xlabel('Range (pixels)')
        axes[0].set_ylabel('Azimuth (pixels)')
        plt.colorbar(im1, ax=axes[0], label='Phase (rad)')
        
        # Coherence
        im2 = axes[1].imshow(self.coherence_stack[ifg_index, :, :], cmap='hot')
        axes[1].set_title(f'Coherence\n{title_str}')
        axes[1].set_xlabel('Range (pixels)')
        axes[1].set_ylabel('Azimuth (pixels)')
        plt.colorbar(im2, ax=axes[1], label='Coherence')
        
        # Coherence-masked phase
        phase_masked = np.ma.masked_where(self.coherence_stack[ifg_index, :, :] < 0.5, 
                                          self.phase_stack[ifg_index, :, :])
        im3 = axes[2].imshow(phase_masked, cmap=cmap)
        axes[2].set_title(f'Phase (Coherence > 0.5)\n{title_str}')
        axes[2].set_xlabel('Range (pixels)')
        axes[2].set_ylabel('Azimuth (pixels)')
        plt.colorbar(im3, ax=axes[2], label='Phase (rad)')
        
        plt.tight_layout()
        return fig, axes

print("GUNWTimeSeries class defined successfully!")

<hr>

(gunw-ts-analysis)=
## 8. Basic Time-Series Analysis

### 8a. Time-series statistics and deformation analysis

In [ ]:
def compute_time_series_statistics(gunw_ts_object):
    """
    Compute basic statistics on loaded interferogram stack.
    """
    if gunw_ts_object.phase_stack is None:
        print("No data loaded")
        return None
    
    n_ifg = gunw_ts_object.phase_stack.shape[0]
    
    stats = {
        'mean_phase': np.nanmean(gunw_ts_object.phase_stack, axis=(1, 2)),
        'std_phase': np.nanstd(gunw_ts_object.phase_stack, axis=(1, 2)),
        'mean_coherence': np.nanmean(gunw_ts_object.coherence_stack, axis=(1, 2)),
    }
    
    print(f"Time-series statistics for {n_ifg} interferograms:\n")
    for i, (ref_date, sec_date) in enumerate(gunw_ts_object.dates):
        print(f"Interferogram {i+1}: {ref_date.date()} - {sec_date.date()}")
        print(f"  Mean phase: {stats['mean_phase'][i]:.4f} rad")
        print(f"  Std phase: {stats['std_phase'][i]:.4f} rad")
        print(f"  Mean coherence: {stats['mean_coherence'][i]:.4f}")
        print()
    
    return stats

def phase_to_los_displacement(phase_rad, wavelength=NISAR_L_WAVELENGTH):
    """
    Convert unwrapped phase to line-of-sight (LOS) displacement.
    
    LOS displacement = -λ / (4π) × phase
    
    Positive displacement = motion away from sensor
    Negative displacement = motion toward sensor
    
    Parameters:
    -----------
    phase_rad : float or np.ndarray
        Phase in radians
    wavelength : float
        Radar wavelength (default: NISAR L-band)
    
    Returns:
    --------
    float or np.ndarray : LOS displacement in meters
    """
    return -wavelength / (4 * np.pi) * phase_rad

print("Analysis functions defined successfully!")

<hr>

(gunw-ts-summary)=
## 9. Summary and Next Steps

### What you've learned:
1. **Search and Download GUNW Data**: Used asf_search to find and download NISAR GUNW products by track, date range, and AOI
2. **NISAR GUNW Product Structure**: Accessed unwrapped phase, coherence, and ancillary data layers
3. **Atmospheric Corrections**: Understood tropospheric delay effects and how to apply TROPO corrections
4. **Time-Series Analysis**: Built stacks of interferograms and computed time-series statistics
5. **Phase-to-Displacement Conversion**: Related InSAR phase measurements to ground deformation

### Next Steps:
1. **Advanced Time-Series Inversion**: Use tools like MintPy for small-baseline time-series analysis
2. **Atmospheric Phase Screening**: Apply additional filtering to remove residual atmospheric noise
3. **Multi-Frequency Analysis**: If available, combine L-band and S-band data for better ionosphere correction
4. **Spatial Filtering**: Apply high-pass or band-pass filters to separate deformation signals
5. **Deformation Rate Estimation**: Compute linear deformation rates over time
6. **Error Propagation**: Assess uncertainties in displacement measurements

### Useful Resources:
- NISAR Data User Guide: https://nisar-docs.asf.alaska.edu/
- ASF Search Documentation: https://nisar-docs.asf.alaska.edu/asf-search/
- ARIA-tools Documentation: https://github.com/aria-tools/ARIA-tools
- MintPy (time-series software): https://github.com/insarlab/MintPy
- OPERA TROPO Product: https://www.jpl.nasa.gov/go/opera/products/tropo-product/

<hr>

(gunw-ts-references)=
## 10. References

### NISAR Documentation
- NISAR Data User Guide: https://nisar-docs.asf.alaska.edu/
- NISAR GUNW Product Specification: Available through NASA NISAR project
- ASF Search API: https://nisar-docs.asf.alaska.edu/asf-search/

### InSAR Theory and Methods
- Bürgmann, R., Rosen, P. A., & Fielding, E. J. (2000). Synthetic aperture radar interferometry to measure Earth's surface topography and its deformation. Annual Review of Earth and Planetary Sciences, 28, 169-209.
- Massonnet, D., & Feigl, K. L. (1998). Radar interferometry and its application to changes in the Earth's surface. Reviews of Geophysics, 36(4), 441-500.

### Atmospheric Corrections
- Bekaert, D. P. S., et al. (2015). A spatially variable power law tropospheric correction technique. Journal of Geophysical Research, 120, 1058-1074.
- OPERA TROPO Product Documentation: https://www.jpl.nasa.gov/go/opera/products/tropo-product/

### Related Software and Tools
- ARIA-tools: https://github.com/aria-tools/ARIA-tools
- MintPy: https://github.com/insarlab/MintPy
- ISCE3: https://github.com/isce-framework/isce3
- asf_search: https://github.com/asfadmin/asf-search-py
- rioxarray: https://corteva.github.io/rioxarray/